# LCEL and Runnables

Assistant version 3: Analyze sentiment, extract themes and suggest replies in parallel with a cheaper fallback.

Read `README.md` for the analogies, diagrams, API explanations, common mistakes and quiz. These cells contain the same source as the scripts; edit inputs and rerun a cell to explore. Outputs are intentionally cleared in Git. This notebook executes scripted fixtures with network blocked, not real models. Use the script commands without `--offline` for live experiments.

Start with `uv sync --group authoring`, then `uv run jupyter lab` from the repository root.

In [ ]:
import asyncio
import os
import sys
from pathlib import Path

root = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").is_file()
)
os.chdir(root)
sys.path.insert(0, str(root))
os.environ["COURSE_OFFLINE"] = "true"
os.environ["LANGSMITH_TRACING"] = "false"
from common.testing import offline_environment

print("OFFLINE FIXTURES: no API fees; outputs do not measure model quality.")

## Runnable interface; pipe operator; RunnableSequence

Predict the output before running. The function definitions below match `examples/01_sequence.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Compose prompt, model and text parser with the LCEL pipe operator.

Run: uv run python modules/04_lcel_runnables/examples/01_sequence.py --offline
Expected: A RunnableSequence and a brief Brightwave reply.
Cost: One short hosted call, typically under $0.01; --offline is free.
"""

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableSequence

from common.errors import run_cli
from common.lesson import lesson_settings
from common.models import get_chat_model
from common.pretty import print_text


def main() -> None:
    settings = lesson_settings()
    prompt = ChatPromptTemplate.from_messages(
        [("system", "Write one Brightwave greeting."), ("human", "{customer}")]
    )
    # KEY IDEA: each stage's output becomes the next stage's input.
    pipeline = prompt | get_chat_model(settings=settings) | StrOutputParser()
    assert isinstance(pipeline, RunnableSequence)
    print_text(pipeline.invoke({"customer": "Avery Demo"}))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## RunnableParallel; fan-out and merged results

Predict the output before running. The function definitions below match `examples/02_parallel.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Fan the same review out into named Runnable branches.

Run: uv run python modules/04_lcel_runnables/examples/02_parallel.py --offline
Expected: A dictionary with length and words, both computed from the same input.
Cost: Pure Python; no model calls.
"""

from langchain_core.runnables import RunnableLambda, RunnableParallel

from common.data import load_csv
from common.errors import run_cli
from common.lesson import lesson_settings
from common.pretty import print_text


def main() -> None:
    lesson_settings()
    pipeline = RunnableParallel(
        length=RunnableLambda(len), words=RunnableLambda(lambda text: len(text.split()))
    )
    # KEY IDEA: branches receive the same input, not another branch's output.
    print_text(str(pipeline.invoke(load_csv("reviews.csv")[0]["text"])))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## RunnablePassthrough; assign; preserve original inputs

Predict the output before running. The function definitions below match `examples/03_passthrough.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Preserve original fields and enrich an input dictionary with assign.

Run: uv run python modules/04_lcel_runnables/examples/03_passthrough.py --offline
Expected: The original review_id/text plus character count.
Cost: No model calls.
"""

from langchain_core.runnables import RunnablePassthrough

from common.errors import run_cli
from common.lesson import lesson_settings
from common.pretty import print_text


def main() -> None:
    lesson_settings()
    pipeline = RunnablePassthrough.assign(characters=lambda row: len(row["text"]))
    # KEY IDEA: assign adds keys to a dictionary; passthrough alone returns its input.
    print_text(str(pipeline.invoke({"review_id": "V0001", "text": "Comfortable tent."})))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## RunnableLambda; @chain; adapting ordinary Python

Predict the output before running. The function definitions below match `examples/04_python_functions.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Adapt ordinary Python with RunnableLambda and the chain decorator.

Run: uv run python modules/04_lcel_runnables/examples/04_python_functions.py --offline
Expected: A clean Brightwave SKU and its catalogue label.
Cost: No model calls.
"""

from langchain_core.runnables import RunnableLambda, chain

from common.errors import run_cli
from common.lesson import lesson_settings
from common.pretty import print_text


def normalize(value: str) -> str:
    return value.strip().upper()


@chain
def label(sku: str) -> str:
    return f"Brightwave product: {sku}"


def main() -> None:
    lesson_settings()
    # KEY IDEA: wrapping gives existing functions invoke/batch/config support.
    print_text((RunnableLambda(normalize) | label).invoke("  bw-001  "))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Routing and branching; plain Python versus LCEL versus LangGraph

Predict the output before running. The function definitions below match `examples/05_routing.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Route a validated ticket using explicit business rules.

Run: uv run python modules/04_lcel_runnables/examples/05_routing.py --offline
Expected: high -> human review; low -> standard queue.
Cost: No model calls.
"""

from langchain_core.runnables import RunnableBranch, RunnableLambda

from common.errors import run_cli
from common.lesson import lesson_settings
from common.pretty import print_text


def router() -> RunnableBranch:
    return RunnableBranch(
        (lambda row: row["priority"] == "high", RunnableLambda(lambda _: "human review")),
        RunnableLambda(lambda _: "standard queue"),
    )


def main() -> None:
    lesson_settings()
    # KEY IDEA: deterministic routing is often clearer than asking another model.
    print_text(str(router().batch([{"priority": "high"}, {"priority": "low"}])))
    print_text(
        "Use Python for simple conditions, LCEL for composable dataflow, "
        "LangGraph for durable loops."
    )


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## with_fallbacks, with_retry; failure then recovery

Predict the output before running. The function definitions below match `examples/06_resilience.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Demonstrate a bounded retry and a fallback with deliberate failures.

Run: uv run python modules/04_lcel_runnables/examples/06_resilience.py --offline
Expected: Retry succeeds on attempt 2; a permanent failure uses the alternate path.
Cost: No API calls; only deliberate local failures.
"""

from collections.abc import Callable

from langchain_core.runnables import RunnableLambda

from common.errors import run_cli
from common.lesson import lesson_settings
from common.pretty import print_text


def transient_request() -> Callable[[str], str]:
    attempts = 0

    def request(text: str) -> str:
        nonlocal attempts
        attempts += 1
        if attempts == 1:
            raise TimeoutError("Synthetic temporary timeout")
        return f"Recovered on attempt {attempts}: {text}"

    return request


def unavailable(text: str) -> str:
    raise TimeoutError("Synthetic service outage")


def main() -> None:
    lesson_settings()
    retry = RunnableLambda(transient_request()).with_retry(
        retry_if_exception_type=(TimeoutError,), stop_after_attempt=2, wait_exponential_jitter=False
    )
    print_text(retry.invoke("Brightwave review"))
    # KEY IDEA: retries repeat one path; fallbacks invoke an alternative implementation.
    alternate = RunnableLambda(unavailable).with_fallbacks(
        [RunnableLambda(lambda _: "Queue for human review")], exceptions_to_handle=(TimeoutError,)
    )
    print_text(alternate.invoke("Brightwave review"))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## with_config tags, metadata, run_name; batch max_concurrency; async

Predict the output before running. The function definitions below match `examples/07_config_batch.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Carry tags/metadata/run names through a bounded batch and asynchronous call.

Run: uv run python modules/04_lcel_runnables/examples/07_config_batch.py --offline
Expected: Three normalized SKUs followed by one awaited SKU.
Cost: No API calls; do not place secrets or ticket text in trace metadata.
"""

import asyncio

from langchain_core.runnables import Runnable, RunnableLambda

from common.errors import run_cli
from common.lesson import lesson_settings
from common.pretty import print_text


async def async_example(pipeline: Runnable) -> None:
    print_text(await pipeline.ainvoke(" bw-004 "))


def main() -> None:
    lesson_settings()
    pipeline = RunnableLambda(lambda sku: sku.strip().upper()).with_config(
        tags=["brightwave", "foundation"], metadata={"lesson": "04"}, run_name="normalize_sku"
    )
    # KEY IDEA: config controls execution/tracing; it is not a prompt variable.
    print_text(str(pipeline.batch(["bw-001", "bw-002", "bw-003"], config={"max_concurrency": 2})))
    asyncio.run(async_example(pipeline))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## astream_events; get_graph().draw_mermaid()

Predict the output before running. The function definitions below match `examples/08_events_diagram.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Inspect runnable lifecycle events and a locally generated Mermaid graph.

Run: uv run python modules/04_lcel_runnables/examples/08_events_diagram.py --offline
Expected: Mermaid text plus on_chain_start/on_chain_stream/on_chain_end events.
Cost: No model or rendering-service calls; Mermaid text is generated locally.
"""

import asyncio

from langchain_core.runnables import Runnable, RunnableLambda

from common.errors import run_cli
from common.lesson import lesson_settings
from common.pretty import print_text


def pipeline() -> Runnable:
    return RunnableLambda(lambda text: text.strip(), name="trim") | RunnableLambda(
        str.upper, name="uppercase"
    )


async def show_events() -> None:
    async for event in pipeline().astream_events(" bw-001 ", version="v2"):
        print_text(f"{event['event']}: {event['name']}")


def main() -> None:
    lesson_settings()
    # KEY IDEA: lifecycle events describe execution; they are not all model text tokens.
    print_text(pipeline().get_graph().draw_mermaid())
    asyncio.run(show_events())


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Real-world use case: Analyze sentiment, extract themes and suggest replies in parallel with a cheaper fallback

Predict the output before running. The function definitions below match `use_case/review_insights.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Run sentiment, theme and reply branches over reviews with a low-cost fallback.

Run: uv run python modules/04_lcel_runnables/use_case/review_insights.py --offline
Expected: Three review reports by default; use --all to process all 300 reviews.
Cost: Default nine calls, usually under $0.02; --all is 900 calls, budget $1; offline/local no
API fee.
"""

import argparse
import json

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import Runnable, RunnableLambda, RunnableParallel

from common.config import ROOT, Settings, get_settings
from common.data import load_csv
from common.errors import run_cli
from common.models import get_chat_model
from common.pretty import print_text
from common.resilience import TRANSIENT_ERRORS


def build_pipeline(settings: Settings) -> Runnable:
    primary_settings = settings.model_copy(update={"model_max_retries": 0})
    cheap_settings = primary_settings.model_copy(
        update={"llm_model": getattr(settings, f"{settings.llm_provider}_chat_model")}
    )
    branches = {}
    for task, fixture in {
        "sentiment": "positive",
        "theme": "comfort",
        "reply": "Thank you for sharing your feedback.",
    }.items():
        prompt = ChatPromptTemplate.from_messages(
            [
                (
                    "system",
                    f"For this Brightwave review, return only {task}. "
                    "Do not invent product facts or promise compensation.",
                ),
                ("human", "{text}"),
            ]
        )
        primary = get_chat_model(settings=primary_settings, responses=[fixture] * 300)
        fallback = get_chat_model(settings=cheap_settings, responses=[fixture] * 300)
        # Retry only transient failures; schema/programming errors should remain visible.
        branches[task] = (
            prompt
            | primary.with_fallbacks([fallback], exceptions_to_handle=TRANSIENT_ERRORS)
            | StrOutputParser()
        )
    return RunnableLambda(lambda row: {"text": row["text"]}) | RunnableParallel(**branches)


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--offline", action="store_true")
    parser.add_argument("--all", action="store_true", help="Process all 300 rows (900 model calls)")
    args = parser.parse_args()
    settings = get_settings()
    if args.offline:
        settings = settings.model_copy(update={"course_offline": True})
    rows = load_csv("reviews.csv")
    selected = rows if args.all else rows[:3]
    results = build_pipeline(settings).batch(selected, config={"max_concurrency": 2})
    output = ROOT / settings.runtime_dir / "review_insights.jsonl"
    output.parent.mkdir(parents=True, exist_ok=True)
    reports = [
        {"review_id": row["review_id"], **result}
        for row, result in zip(selected, results, strict=True)
    ]
    output.write_text("".join(json.dumps(report) + "\n" for report in reports), encoding="utf-8")
    print_text(
        "OFFLINE FIXTURE: outputs do not classify the input."
        if settings.course_offline
        else "LIVE review analysis"
    )
    print_text(
        f"Fallback profile: {getattr(settings, f'{settings.llm_provider}_chat_model')}; "
        "cheaper only when your LLM_MODEL is a more expensive choice."
    )
    print_text(json.dumps(reports, indent=2))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Practice and reflect

Change a Brightwave input and rerun the relevant cell. Inspect whether the prompt, tool arguments or parser result changed. A scripted model reply may stay the same: that is a fixture limitation, not evidence that a real model ignores input. Complete the three exercises in `exercises/`, run their target checks, and answer the quiz in `quiz.md`. Leave outputs cleared before committing.

Run `uv run pytest modules/04_lcel_runnables/tests` for the supplied solution checks.